# DSORNet-v3.1 — Google Colab GPU

This notebook runs DSORNet-v3.1 on an NVIDIA Colab GPU using the same `dsorn_v31.py` code path as the repository.

Before running the notebook, select **Runtime → Change runtime type → T4 GPU** or another available NVIDIA GPU.


## 1. Verify the Colab CUDA runtime


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not enabled. In Colab choose Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)


## 2. Clone DSOR and install Colab-safe dependencies

Colab already ships a CUDA-enabled PyTorch build. The Colab requirements file deliberately does not reinstall PyTorch.


In [ ]:
!rm -rf /content/DSOR
!git clone -q https://github.com/godsonj64/DSOR.git /content/DSOR
%cd /content/DSOR
!python -m pip install -q -r requirements-colab.txt


## 3. Run the repository checks


In [ ]:
!pytest -q


## 4. Verify DSORNet executes on the GPU


In [ ]:
import torch
from dsorn_v31 import DSORNetV31Sequential

device = torch.device("cuda")
model = DSORNetV31Sequential(num_classes=10).to(device)
x = torch.randn(2, 3, 32, 32, device=device)

with torch.no_grad():
    logits = model(x)

print("Device:", next(model.parameters()).device)
print("Output shape:", tuple(logits.shape))
print("Finite:", bool(torch.isfinite(logits).all()))
print("Allocated GPU memory (MiB):", round(torch.cuda.memory_allocated() / 2**20, 2))


## 5. Configure a CIFAR run

The defaults below are deliberately small enough for an initial research run. Increase the sample count and epoch count for a more substantial experiment.


In [ ]:
DATASET = "cifar10"       # "cifar10" or "cifar100"
TRAIN_PER_CLASS = 100
TEST_PER_CLASS = 50
EPOCHS = 12
BATCH_SIZE = 128

print({
    "dataset": DATASET,
    "train_per_class": TRAIN_PER_CLASS,
    "test_per_class": TEST_PER_CLASS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
})


## 6. Train DSORNet-v3.1 on CUDA


In [ ]:
import subprocess

cmd = [
    "python", "dsorn_v31.py",
    "--dataset", DATASET,
    "--data-dir", "/content/data",
    "--train-per-class", str(TRAIN_PER_CLASS),
    "--test-per-class", str(TEST_PER_CLASS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--device", "cuda",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Optional CIFAR-100 configuration

For CIFAR-100, a smaller per-class sample count is usually more practical for a Colab pilot.


In [ ]:
# Example:
# DATASET = "cifar100"
# TRAIN_PER_CLASS = 25
# TEST_PER_CLASS = 20
# EPOCHS = 15
# BATCH_SIZE = 128
#
# Re-run the training cell after changing these values.


## Notes

- The model contains no convolution modules.
- CUDA uses the same PyTorch tensor implementation as the local code; no separate Colab model fork is introduced.
- The small-subset CIFAR commands are research pilots, not full-dataset benchmark claims.
- Colab runtimes are ephemeral, so copy any experiment artifacts you add to Google Drive before the session ends.
